# 28 Local Model Refinement and Ablation

Local-only Phase 28 notebook. This notebook inspects baseline artifacts, runs controlled refinement experiments, runs required feature-group ablations, and writes local outputs.

**Notebook purpose:** Runs controlled refinement experiments (balanced resampling, hyperparameter tuning) and feature-group ablation studies against the Phase 27 baseline. Writes refined metrics and ablation results.

**Required data:** `local/derived/features/bluesky_engagement_features.parquet` (nb 26) and all Phase 27 baseline artifacts under `local/derived/modeling/` (from notebook 27).

**Run order:** Run after notebook 27 (baseline modeling). This is the final modeling notebook.

## 1) Inputs and Baseline Artifacts

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:  # pragma: no cover
    def display(value):
        print(value)


def find_repo_root() -> Path:
    probe = Path.cwd().resolve()
    for candidate in [probe, *probe.parents]:
        if (candidate / 'src' / 'modeling' / 'model_refinement.py').exists():
            return candidate
    raise FileNotFoundError('Could not find repo root containing src/modeling/model_refinement.py')


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.modeling.model_refinement import (
    read_baseline_artifacts,
    run_model_refinement,
    write_refinement_artifacts,
)

FEATURE_PATH = ROOT / 'local/derived/features/bluesky_engagement_features.parquet'
BASELINE_METRICS_PATH = ROOT / 'local/derived/modeling/baseline_model_metrics.json'
BASELINE_SUMMARY_PATH = ROOT / 'local/derived/modeling/baseline_model_summary.json'
BASELINE_CONFUSION_PATH = ROOT / 'local/derived/modeling/baseline_confusion_matrix.csv'
BASELINE_IMPORTANCE_PATH = ROOT / 'local/derived/modeling/baseline_feature_importance.parquet'

_required = [FEATURE_PATH, BASELINE_METRICS_PATH, BASELINE_SUMMARY_PATH, BASELINE_CONFUSION_PATH, BASELINE_IMPORTANCE_PATH]
_missing = [str(p) for p in _required if not p.exists()]
if _missing:
    raise FileNotFoundError(
        "DATA NOT YET AVAILABLE -- run notebook 27 (baseline modeling) first.\n"
        "Missing: " + ", ".join(_missing)
    )

feature_df = pd.read_parquet(FEATURE_PATH)
baseline = read_baseline_artifacts(
    metrics_path=BASELINE_METRICS_PATH,
    summary_path=BASELINE_SUMMARY_PATH,
    confusion_path=BASELINE_CONFUSION_PATH,
    importance_path=BASELINE_IMPORTANCE_PATH,
)

print('Feature shape:', feature_df.shape)
print('Baseline best model:', baseline['metrics']['best_model_name'])
print('Baseline target:', baseline['metrics']['target_column'])

Feature shape: (19999, 92)
Baseline best model: bagged_stump_ensemble
Baseline target: engagement_label


## 2) Run Controlled Refinement

In [2]:
modeling_output = run_model_refinement(feature_df=feature_df, baseline_artifacts=baseline)

experiments_df = pd.DataFrame(modeling_output['refinement_experiments'])
display(experiments_df[[
    'experiment_name',
    'model_family',
    'balanced_resample',
    'accuracy',
    'balanced_accuracy',
    'macro_f1',
]].sort_values('experiment_name', kind='stable').reset_index(drop=True))

print('Selected refined experiment:', modeling_output['selected_refined_experiment_name'])
print('Recommendation:', modeling_output['comparison_to_baseline']['recommended_final_story'])


,experiment_name,model_family,balanced_resample,accuracy,balanced_accuracy,macro_f1
0,bagged_stump_balanced_resample,bagged_stump,True,0.545258,0.396041,0.318225
1,bagged_stump_balanced_resample_tuned,bagged_stump,True,0.550092,0.402936,0.325521
2,control_bagged_stump,bagged_stump,False,0.856309,0.333333,0.307531
3,softmax_refined_control,softmax,False,0.406568,0.416508,0.292307


Selected refined experiment: bagged_stump_balanced_resample_tuned
Recommendation: refined


## 3) Feature-Group Ablation Results

In [3]:
ablation_df = modeling_output['model_ablation_results']
display(ablation_df[[
    'experiment_name',
    'removed_feature_groups',
    'n_features_used',
    'skipped',
    'skipped_reason',
    'accuracy',
    'balanced_accuracy',
    'macro_f1',
]].sort_values('experiment_name', kind='stable').reset_index(drop=True))

print('Derived feature groups:')
for group_name, cols in modeling_output['feature_groups'].items():
    print(f'  {group_name}: {len(cols)}')


,experiment_name,removed_feature_groups,n_features_used,skipped,skipped_reason,accuracy,balanced_accuracy,macro_f1
0,full_features,[],56,False,None,0.550092,0.402936,0.325521
1,no_actor_account_features,[actor_account],37,False,None,0.398566,0.388638,0.276370
2,no_post_text_features,[post_text],41,False,None,0.560093,0.401595,0.323192
3,no_temporal_features,[temporal],52,False,None,0.553259,0.405378,0.326595
4,no_trend_match_features,[trend_match],34,False,None,0.562760,0.399915,0.322280


Derived feature groups:
  trend_match: 14
  post_text: 15
  actor_account: 15
  temporal: 4


## 4) Confusion Matrix and Error Follow-Up

In [4]:
comparison = modeling_output['comparison_to_baseline']
print('Metric deltas vs baseline:', comparison['metric_deltas'])

conf_df = modeling_output['refined_confusion_matrix']
display(conf_df)

pred_df = modeling_output['refined_prediction_sample']
display(pred_df[['uri', 'true_label', 'predicted_label', 'is_correct']])


Metric deltas vs baseline: {'macro_f1_delta': 0.017989974925728902, 'balanced_accuracy_delta': 0.06960305927050259, 'accuracy_delta': -0.30621770295049167, 'HIGH_recall_delta': 0.5127551020408163}


,HIGH,LOW,MEDIUM
HIGH,201,167,24
LOW,1703,3051,383
MEDIUM,215,207,48


,uri,true_label,predicted_label,is_correct
0,at://did:plc:222p42fegwhwfyrc3gqam76j/app.bsky...,LOW,HIGH,False
1,at://did:plc:22bixok3zcw6dv72gyi5pwox/app.bsky...,LOW,LOW,True
2,at://did:plc:22ezkuvas6f545oal47snp5x/app.bsky...,LOW,HIGH,False
3,at://did:plc:22hp52c2fr5ywmvqhfomzqja/app.bsky...,LOW,MEDIUM,False
4,at://did:plc:22k3roaumaq77xwhhpdcjdka/app.bsky...,LOW,LOW,True
...,...,...,...,...
5994,at://did:plc:zzififr6febsnsu754pidsxi/app.bsky...,HIGH,LOW,False
5995,at://did:plc:zzififr6febsnsu754pidsxi/app.bsky...,LOW,LOW,True
5996,at://did:plc:zzqefjgksgaux7s6sghipwsh/app.bsky...,LOW,LOW,True
5997,at://did:plc:zzvyx5nsscf23fso2rcvtd5a/app.bsky...,LOW,LOW,True


## 5) Write Phase 28 Outputs

In [5]:
artifact_paths = write_refinement_artifacts(
    modeling_output=modeling_output,
    output_dir=str(ROOT / 'local/derived/modeling'),
    sample_csv_path=str(ROOT / 'data/samples/refined_predictions_sample_1000.csv'),
)
artifact_paths

{'metrics_json': '/Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/local/derived/modeling/refined_model_metrics.json',
 'ablation_parquet': '/Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/local/derived/modeling/model_ablation_results.parquet',
 'feature_importance_parquet': '/Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/local/derived/modeling/refined_feature_importance.parquet',
 'prediction_sample_parquet': '/Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/local/derived/modeling/refined_predictions_sample.parquet',
 'prediction_sample_csv': '/Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/data/samples/refined_predictions_sample_1000.csv',
 'summary_json': '/Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/Appl